In [1]:
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

transaction_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund\Transaction\Transaction_refund.csv"
df = pd.read_csv(transaction_path, low_memory=False)




In [2]:
df=df[df['marketplace_id'] !="marketplace_id"]
df = df[df['refund_init_date'] !="refund_init_date"]
df = df[df['rf_orders'] !="rf_orders"]
df['refund_init_date'] =df['refund_init_date'].astype("datetime64[ns]")
df['rf_orders'] = pd.to_numeric(df["rf_orders"], errors ='coerce')
df['p_refund_count'] = pd.to_numeric(df["p_refund_count"], errors ='coerce')
df['is_shopsy_order'] = df['is_shopsy_order'].astype(str).str.lower()
df=df.sort_values(by=["refund_init_date"])
df[df['rf_orders'].isna()]
#print(df)
print(df['is_shopsy_order'].unique())
print(df['marketplace_id'].unique())


['true' 'false' 'nan']
['FLIPKART' 'HYPERLOCAL' nan 'GROCERY' 'RCBP']


In [3]:
df= df[(df["marketplace_id"].isin(["HYPERLOCAL"])) &
                          (~df["is_shopsy_order"].isin(["true",'True']))].copy()


filtered_date = df[df['refund_init_date'].isin(['2026-01-01', '2026-01-02'])]

refund_final_status =(
    filtered_date.groupby(['refund_final_status_updated','refund_init_date'],observed=False)["rf_orders"]
    .sum()
    .reset_index()
    .rename(columns={"rf_orders": "refund_count"})
    .sort_values(by=['refund_init_date']))
print(refund_final_status)



Empty DataFrame
Columns: [refund_final_status_updated, refund_init_date, refund_count]
Index: []


C:\Users\rajeshkumar.t\AppData\Local\Temp\ipykernel_21540\2334809066.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  filtered_date = df[df['refund_init_date'].isin(['2026-01-01', '2026-01-02'])]


In [4]:
df= df[(df["marketplace_id"].isin(["HYPERLOCAL"])) &
                          (~df["is_shopsy_order"].isin(["true","True"]))].copy()

within_24 = ["2. 0 To 4 Hrs", "3. 4 To 8 Hrs", "4. 8 To 24 Hrs"]
df["refund_init_completion_bucket_flag"]=df["refund_init_completion_bucket"].apply(
    lambda x: "within_24" if x in within_24 else "greater_24"
)

transaction_order = ['Completed', 'Failed', 'Cancelled', 'Stuck', 'CLONED']
df["refund_final_status_updated"] = pd.Categorical(df["refund_final_status_updated"], categories=transaction_order, ordered=True)

df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

refund_final_status = (df.groupby(["refund_final_status_updated","refund_init_completion_bucket_flag", "refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "refund_status_count"}))

refund_final_status = (
    df.groupby(["refund_final_status_updated", "refund_init_completion_bucket_flag", "refund_init_date"], observed=False)["rf_orders"]
    .sum()
    .reset_index()
    .rename(columns={"rf_orders": "refund_status_count"})
)
pivot_source = refund_final_status.pivot_table(index=["refund_final_status_updated","refund_init_completion_bucket_flag"],
                                                columns = "refund_init_date",
                                                values = "refund_status_count",
                                                aggfunc='sum',
                                                observed= False
                                              ).fillna(0)
                                              
pivot_source.columns=pd.to_datetime(pivot_source.columns, errors='coerce')
pivot_source = pivot_source.sort_index(axis=1)
pivot_source.columns = pivot_source.columns.strftime('%d-%m-%Y')
pivot_source.reset_index(inplace=True)

#print(pivot_source)

####-------------------------###

df["rf_orders"] = pd.to_numeric(df["rf_orders"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')


transaction_order5 = ['Completed', 'Failed', 'Cancelled', 'Stuck', 'CLONED']
df["refund_final_status_updated"] = pd.Categorical(df["refund_final_status_updated"], categories=transaction_order, ordered=True)

df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')
refund_final_status1 = (df.groupby(["refund_final_status_updated","refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "refund_status_count1"}))

refund_final_status1 = (
    df.groupby(["refund_final_status_updated",  "refund_init_date"], observed=False)["rf_orders"]
    .sum()
    .reset_index()
    .rename(columns={"rf_orders": "refund_status_count1"})
)
pivot_source1 = refund_final_status1.pivot_table(index=["refund_final_status_updated"],
                                                columns = "refund_init_date",
                                                values = "refund_status_count1",
                                                aggfunc='sum',
                                                observed= False
                                              ).fillna(0)
                                              
pivot_source1.columns=pd.to_datetime(pivot_source1.columns, errors='coerce')
pivot_source1 = pivot_source1.sort_index(axis=1)
pivot_source1.columns = pivot_source1.columns.strftime('%d-%m-%Y')
pivot_source1.reset_index(inplace=True)

#print(pivot_source1)


##------------------##
##refund_reason_flag
df["refund_reason_group"] = df["refund_reason_flag"].apply(
    lambda x: "Cancel_RTO" if x in ["Cancellation", "RTO"] 
    else x if x in ["Return","Adonc"]
    else "Others"
)

df["rf_orders"] = pd.to_numeric(df["rf_orders"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

transaction_order1 = ['Cancel_RTO', 'Return', 'Adonc', 'Others']
df["refund_reason_group"] = pd.Categorical(df["refund_reason_group"], categories=transaction_order1, ordered=True)

refund_reason_flag1 = (df.groupby(["refund_reason_group", "refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "refund_reason_group_count"}))

pivot_source_reason = refund_reason_flag1.pivot_table(index="refund_reason_group",
                                                columns = "refund_init_date",
                                                values = "refund_reason_group_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_reason.columns=pd.to_datetime(pivot_source_reason.columns, errors='coerce')
pivot_source_reason = pivot_source_reason.sort_index(axis=1)
pivot_source_reason.columns = pivot_source_reason.columns.strftime('%d-%m-%Y')
pivot_source_reason.reset_index(inplace=True)

#print(pivot_source_reason)

##------------------##
##refund_reason_flag
df["payment_type_final_flag"] = df["payment_type_final"].apply(
    lambda x: "Pre-paid" if x in ["Pre-paid", "EGV/Wallet/Coin"] 
    else x 
)
transaction_order2 = ['Pre-paid', 'COD', 'Multi-payment', 'NA']
df["payment_type_final_flag"] = pd.Categorical(df["payment_type_final_flag"], categories=transaction_order2, ordered=True)

refund_summary_by_payment_type = (df.groupby(["payment_type_final_flag", "refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "payment_type_final_count"}))

pivot_source_payment = refund_summary_by_payment_type.pivot_table(index="payment_type_final_flag",
                                                columns = "refund_init_date",
                                                values = "payment_type_final_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_payment.columns=pd.to_datetime(pivot_source_payment.columns, errors='coerce')
pivot_source_payment = pivot_source_payment.sort_index(axis=1)
pivot_source_payment.columns = pivot_source_payment.columns.strftime('%d-%m-%Y')
pivot_source_payment.reset_index(inplace=True)

##print(pivot_source_payment)

##------------------##
##ARN_issue_flag

transaction_order3 = ['ARN_GENERATED', 'ARN_NOT_GENERATED', 'ARN_NOT_APPLICABLE']
df["ARN_ISSUE_FLAG"] = pd.Categorical(df["ARN_ISSUE_FLAG"], categories=transaction_order3, ordered=True)

ARN_issue_flag_summary = (df.groupby(["ARN_ISSUE_FLAG", "refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "ARN_ISSUE_count"}))

pivot_source_ARN = ARN_issue_flag_summary.pivot_table(index="ARN_ISSUE_FLAG",
                                                columns = "refund_init_date",
                                                values = "ARN_ISSUE_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_ARN.columns=pd.to_datetime(pivot_source_ARN.columns, errors='coerce')
pivot_source_ARN = pivot_source_ARN.sort_index(axis=1)
pivot_source_ARN.columns = pivot_source_ARN.columns.strftime('%d-%m-%Y')
pivot_source_ARN.reset_index(inplace=True)

##print(pivot_source_ARN)

##------------------##
##Refund_overall


Refund_overall_summary = (df.groupby(["refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "Overall_refund_count"}))

Refund_overall_summary = Refund_overall_summary.sort_values("refund_init_date")
pivot_Refund_overall_summary=Refund_overall_summary.set_index("refund_init_date").T
                                              
pivot_Refund_overall_summary.columns=pd.to_datetime(pivot_Refund_overall_summary.columns, errors='coerce')
pivot_Refund_overall_summary = pivot_Refund_overall_summary.sort_index(axis=1)
pivot_Refund_overall_summary.columns = pivot_Refund_overall_summary.columns.strftime('%d-%m-%Y')
pivot_Refund_overall_summary.reset_index(inplace=True)

#print(pivot_Refund_overall_summary)

####
##Refund_overll_refund_id

##Refund_overall


Refund_overall_summary1 = (df.groupby(["refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "Overall_refund_count1"}))

Refund_overall_summary1 = Refund_overall_summary1.sort_values("refund_init_date")
pivot_Refund_overall_summary1=Refund_overall_summary1.set_index("refund_init_date").T
                                              
pivot_Refund_overall_summary1.columns=pd.to_datetime(pivot_Refund_overall_summary1.columns, errors='coerce')
pivot_Refund_overall_summary1 = pivot_Refund_overall_summary1.sort_index(axis=1)
pivot_Refund_overall_summary1.columns = pivot_Refund_overall_summary1.columns.strftime('%d-%m-%Y')
pivot_Refund_overall_summary1.reset_index(inplace=True)

#print(pivot_Refund_overall_summary1)

##---Void_Transactions-----



##Refund_overall


Refund_void_summary = (df.groupby(["void_flag", "refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "refund_void_count"}))

pivot_void_summary = Refund_void_summary.pivot_table(index="void_flag",
                                                columns = "refund_init_date",
                                                values = "refund_void_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_void_summary.columns=pd.to_datetime(pivot_void_summary.columns, errors='coerce')
pivot_void_summary = pivot_void_summary.sort_index(axis=1)
pivot_void_summary.columns = pivot_void_summary.columns.strftime('%d-%m-%Y')
pivot_void_summary.reset_index(inplace=True)

##print(pivot_void_summary)

#####------------------------
##Refund_modes

emi_pg_ids =['cfa_juspay_dmi_emi',
    'cfa_juspay_fibe_emi',
    'cfa_juspay_abfl_emi',
    'cfa_juspay_chola_emi',
    'cfa_juspay_tvs_emi'
]

df["refund_mode"] = df.apply(
lambda row: 'fk_emi' if row['pg_id'] in emi_pg_ids else row['refund_mode'], axis=1)

transaction_order4 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
        "E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","FLIPKART_FINANCE","InvalidRefundMode","LOAN","SUPERPAY_LATER",'SUPERPAY_IN_INSTALLMENT',"fk_emi"]

df["refund_mode"] = df["refund_mode"].apply(
    lambda x: x if x in transaction_order4 else "Others"
)
category_list = transaction_order4 + ["Others"]
df["refund_mode"] = pd.Categorical(df["refund_mode"], categories=category_list, ordered=True)


refund_mode_flag_summary = (df.groupby(["refund_mode", "refund_init_date"], observed=False)["rf_orders"].sum().reset_index()
                .rename(columns={"rf_orders": "refund_mode_count"}))

pivot_source_refund_mode = refund_mode_flag_summary.pivot_table(index="refund_mode",
                                                columns = "refund_init_date",
                                                values = "refund_mode_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_refund_mode.columns=pd.to_datetime(pivot_source_refund_mode.columns, errors='coerce')
pivot_source_refund_mode = pivot_source_refund_mode.sort_index(axis=1)
pivot_source_refund_mode.columns = pivot_source_refund_mode.columns.strftime('%d-%m-%Y')
pivot_source_refund_mode.reset_index(inplace=True)

output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_Hyperlocal_tranaction.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    pivot_source.to_excel(writer, sheet_name ='refund_final_status_updated', index=False)
    pivot_source1.to_excel(writer, sheet_name ='refund_final_status', index=False)
    pivot_source_reason.to_excel(writer, sheet_name ='refund_reason', index=False )
    pivot_source_payment.to_excel(writer, sheet_name ='refund_source_payment', index=False )
    pivot_source_ARN.to_excel(writer, sheet_name = 'refund_ARN_flag', index=False)
    pivot_void_summary.to_excel(writer, sheet_name = 'pivot_Refund_Void', index=False)
    pivot_source_refund_mode.to_excel(writer, sheet_name = 'refund_mode_ssi', index=False)
    pivot_Refund_overall_summary.to_excel(writer, sheet_name ='Overall_rf_count, index =False')
    pivot_Refund_overall_summary1.to_excel(writer, sheet_name ='Overall_p_count1, index =False')
    



print("Excel file successfully saved:",output_path)

Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_Hyperlocal_tranaction.xlsx


In [5]:
raw_df = pd.read_csv(transaction_path, low_memory=False)
print(raw_df.columns.tolist())
print(raw_df["refund_init_date"].head(10))

['refund_year', 'refund_month', 'refund_week', 'refund_init_date', 'refund_complete_date', 'refund_status', 'refund_final_status_updated', 'mr_refund_flag', 'refund_mode', 'auth_state', 'payment_mode', 'instrument_type', 'payment_instrument', 'ref_amount_flag', 'transaction_source', 'pg_id', 'promise_sla_refund_bucket', 'refund_init_completion_bucket', 'overall_ageing_Stuck', 'refund_complition_bucket', 'ARN_ISSUE_FLAG', 'void_flag', 'refund_reason', 'refund_reason_flag', 'flipkart_emi_flag', 'payment_type_final', 'cx_count', 'rf_orders', 'p_refund_count', 'c_refund_count', 'total_refund_amount', 'is_shopsy_order', 'marketplace_id']
0    2025-12-07
1    2025-12-07
2    2025-12-07
3    2025-12-07
4    2025-12-07
5    2025-12-07
6    2025-12-07
7    2025-12-07
8    2025-12-07
9    2025-12-07
Name: refund_init_date, dtype: object


In [6]:
from pandas import MultiIndex

df= df[(df["marketplace_id"].isin(["HYPERLOCAL"])) &
                          (~df["is_shopsy_order"].isin(["true","True"]))].copy()

df["rf_orders"] = pd.to_numeric(df["rf_orders"], errors='coerce')


refund_status_group = {
    'Completed': ['UPI_INTENT', 'CREDIT_CARD', 'COIN', 'IMPS', 'QC_EGV'],
    'Failed': ['IMPS', 'NEFT', 'CREDIT_CARD', 'UPI', 'UPI_INTENT'],
    'Cancelled': ['IMPS', 'NEFT', 'UPI_INTENT', 'CREDIT_CARD_EMI', 'FK_UPI'],
    'Stuck': ['IMPS', 'UPI', 'NEFT', 'UPI_INTENT', 'CREDIT_CARD']
}
refund_status_valid = [mode for group in refund_status_group.values() for mode in group]


df['refund_mode'] =df['refund_mode'].astype(str).str.upper()
valid_pairs = set(
    (status, mode)
    for status, modes in refund_status_group.items()
    for mode in modes
)
df= df[df.apply(lambda x: (x['refund_final_status_updated'], x['refund_mode']) in valid_pairs, axis=1)]

# Step 6: Group
refund_mode_status_latest = (
    df.groupby(['refund_init_date', 'refund_mode', 'refund_final_status_updated'], observed=False)['rf_orders']
    .sum()
    .reset_index()
    .rename(columns={"rf_orders": "Refund_mode_count"})
)


pivot_refund_mode_status_latest = refund_mode_status_latest.pivot_table(
    index=[ "refund_final_status_updated","refund_mode"],
    columns="refund_init_date",
    values="Refund_mode_count",
    aggfunc="sum",
    observed=False
).fillna(0)


# Step 8: Format
pivot_refund_mode_status_latest.columns = pd.to_datetime(pivot_refund_mode_status_latest.columns, errors='coerce').strftime('%d-%m-%Y')
pivot_refund_mode_status_latest.reset_index(inplace=True)

expected_pairs = [
    (status,mode)
    for status, modes in refund_status_group.items()
    for mode in modes
]

expected_index = MultiIndex.from_tuples(expected_pairs, names=['refund_final_status_updated','refund_mode'])
pivot_refund_mode_status_latest.set_index(['refund_final_status_updated','refund_mode'], inplace=True)
pivot_refund_mode_status_latest = pivot_refund_mode_status_latest.reindex(expected_index, fill_value=0)
pivot_refund_mode_status_latest.reset_index(inplace=True)


output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\refund_mode_latest_Hyperlocal_status.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    pivot_refund_mode_status_latest.to_excel(writer, sheet_name='pivot_refund_mode_status', index=False)

print("Excel File Successfully saved:", output_path)


Excel File Successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\refund_mode_latest_Hyperlocal_status.xlsx
